# 🏦 Financial Market Research Agent with Bing Grounding 📈

Welcome to our **Financial Market Research Agent with Bing Grounding** tutorial! In this notebook, we'll demonstrate how to:

1. **Initialize** a project using Microsoft Foundry.
2. **Create an Agent** with the **BingGroundingTool** for real-time web search.
3. **Ask real-world questions** about market trends, interest rates, and financial news.
4. **Retrieve and display** answers with real-time market data and appropriate disclaimers.

### ⚠️ Important Model Support Note ⚠️
> Bing grounding is currently only supported in certain Azure OpenAI models (e.g. `gpt-4o-0513`).
> 
> Make sure you specify a supported model and set the `"x-ms-enable-preview": "true"` header.

## Prerequisites
- Grounding with Bing connection in Microsoft Foundry, which has to be provisioned from Azure portal.
See ["Setup Bing Grounding"](https://learn.microsoft.com/en-us/azure/ai-services/agents/how-to/tools/bing-grounding?tabs=python&pivots=overview#setup) in the documentation for full details.

<img src="./seq-diagrams/bing-connection.png" width="75%"/>

- A `.env` file in the parent directory containing:
  ```bash
  AI_FOUNDRY_PROJECT_ENDPOINT=<your-ai-foundry-project-endpoint>
  AZURE_AI_MODEL_DEPLOYMENT_NAME=<supported-model>
  GROUNDING_WITH_BING_CONNECTION_NAME=<the-name-of-your-bing-connection>
  ```

## Let's Explore Real-Time Financial Research!
We'll integrate **Grounding with Bing** search results into our agent so it can gather current market data, interest rate news, and economic trends. This is perfect for financial advisors who need up-to-date information! 🎉

<br/>



## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login --use-device-code
```

This will provide you with a device code and URL to authenticate in your browser, which is useful for:
- Remote development environments
- Systems without a default browser
- Corporate environments with strict security policies

After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
We'll load environment variables from `.env` and initialize our **AIProjectClient** to manage agents.

In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    BingGroundingSearchConfiguration,
    BingGroundingSearchToolParameters,
    BingGroundingTool,
    PromptAgentDefinition,
)
from azure.identity import InteractiveBrowserCredential
from dotenv import find_dotenv, load_dotenv


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")
load_dotenv(dotenv_path)

tenant_id = os.getenv("TENANT_ID")
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
bing_connection_name = os.getenv("GROUNDING_WITH_BING_CONNECTION_NAME")
required_settings = {
    "TENANT_ID": tenant_id,
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_name,
    "GROUNDING_WITH_BING_CONNECTION_NAME": bing_connection_name,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

credential = InteractiveBrowserCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()
print("AIProjectClient and OpenAI client initialized")

## 2. Create a Bing-Grounded Financial Research Agent

The next cell retrieves the named Grounding with Bing connection from the Foundry project and attaches it to a versioned prompt agent. Ensure the configured model deployment supports Bing grounding.

In [ ]:
def create_bing_grounded_agent():
    """Create a financial research agent backed by Grounding with Bing."""
    try:
        bing_connection = project_client.connections.get(name=bing_connection_name)
    except Exception:
        available_connections = [connection.name for connection in project_client.connections.list()]
        print(f"Available project connections: {available_connections}")
        raise

    bing_tool = BingGroundingTool(
        bing_grounding=BingGroundingSearchToolParameters(
            search_configurations=[
                BingGroundingSearchConfiguration(
                    project_connection_id=bing_connection.id,
                )
            ]
        )
    )

    agent = project_client.agents.create_version(
        agent_name="financial-market-research-agent",
        definition=PromptAgentDefinition(
            model=model_name,
            instructions="""
            You are a financial market research assistant with Bing search capabilities.

            1. Use Bing for current interest rates, market news, and economic trends.
            2. Cite relevant sources and include publication dates when available.
            3. Distinguish sourced facts from your own synthesis.
            4. Do not recommend specific investments or predict market movements.
            5. State that you are not a licensed financial advisor and encourage professional consultation.
            """,
            tools=[bing_tool],
        ),
        description="Financial market research agent with Bing grounding.",
    )
    print(f"Created agent {agent.name}, version {agent.version}")
    return agent


bing_agent = create_bing_grounded_agent()

## 3. Researching Financial Markets 💬
We'll create conversation threads for financial research queries, letting the agent search Bing for current market data. We'll ask about interest rates, Fed policy, and market trends.

In [ ]:
def ask_financial_research_question(agent, user_query):
    """Ask one grounded research question in a new conversation."""
    conversation = openai_client.conversations.create()
    print(f"Researching in conversation {conversation.id}: {user_query}")

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "type": "agent_reference",
                "name": agent.name,
                "version": agent.version,
            }
        },
        input=user_query,
    )
    if not response.output_text:
        raise RuntimeError(f"Response {response.id} did not contain output text.")

    print(response.output_text)
    return conversation.id, response


questions = [
    "What are the current Federal Reserve interest rate decisions and their impact on mortgage rates?",
    "What are the latest trends in banking sector performance and major bank stock movements?",
    "What are financial experts saying about inflation forecasts for the upcoming year?",
]
bing_responses = [
    ask_financial_research_question(bing_agent, question)
    for question in questions
]

## 4. Cleanup & Best Practices
You can optionally delete the agent once you're done. In production, you might keep it around for repeated usage.

### Best Practices
1. **Accuracy** – Bing search results may include disclaimers or partial info. Encourage verification with credible sources.
2. **Bing Query Display** – For compliance with Bing's use and display requirements, show both **website URLs** (in the agent's response) and **Bing search query URLs** (shown above). If the model includes citations, display them as well.
3. **Limits** – Keep an eye on usage, rate limits, or policy constraints for Bing.
4. **Privacy** – Filter search queries to avoid sending sensitive data.
5. **Evaluations** – Use `azure-ai-evaluation` for iterative improvement.


In [ ]:
cleanup_errors = []

for conversation_id, _ in bing_responses:
    try:
        openai_client.conversations.delete(conversation_id)
        print(f"Deleted conversation {conversation_id}")
    except Exception as error:
        cleanup_errors.append(f"conversation {conversation_id}: {error}")

try:
    project_client.agents.delete_version(
        agent_name=bing_agent.name,
        agent_version=bing_agent.version,
    )
    print(f"Deleted agent {bing_agent.name} version {bing_agent.version}")
except Exception as error:
    cleanup_errors.append(f"agent {bing_agent.name} version {bing_agent.version}: {error}")

openai_client.close()
project_client.close()
credential.close()

if cleanup_errors:
    raise RuntimeError("Cleanup failures:\n" + "\n".join(cleanup_errors))

# Congratulations! 🎉

You've successfully completed the **Financial Market Research Agent with Bing Grounding** tutorial! Here's what was accomplished:

## ✅ **What We Built**

### **🌐 Fully Functional Bing-Grounded Agent**
- Created a financial market research agent with **real Bing grounding** capabilities
- Successfully connected to Microsoft Foundry Bing connection
- Agent can now search the web for **real-time financial market information**
- Configured financial-focused instructions with appropriate investment disclaimers

### **🔧 Key Features Demonstrated**

1. **🔗 Successful Bing Connection**
   - **Connected to actual Bing service**: Retrieved connection from Microsoft Foundry
   - **Real web search capabilities**: Agent can now access current information
   - **Proper configuration**: Fixed the `search_configurations` array format
   - **Working end-to-end**: From connection retrieval to agent responses

2. **💬 Advanced Question Processing**
   - Successfully processed financial market queries with **real-time data**:
     - Market trends and stock information (with current data)
     - Interest rate news and economic indicators (latest updates)  
     - Financial news and investment insights

3. **Tool Configuration** - Ensure `search_configurations` is a proper array
4. **Connection Testing** - List available connections when debugging
5. **Resource Management** - Always clean up agents and resources
6. **Financial Disclaimers** - Maintain investment advice disclaimers even with real-time data
7. **Source Transparency** - Leverage web search citations for credibility

## 🔧 **Troubleshooting Guide**

**If Bing grounding fails:**
1. ✅ Check `GROUNDING_WITH_BING_CONNECTION_NAME` in `.env` file
2. ✅ Use `name=` parameter in `connections.get()` call
3. ✅ Ensure `search_configurations` is an array with exactly 1 element
4. ✅ Verify the Bing connection exists in Microsoft Foundry
5. ✅ List available connections for debugging

**For connection errors:**
1. ✅ Use `project_client.connections.list()` to see available connections
2. ✅ Verify connection IDs match expected format
3. ✅ Check Microsoft Foundry portal for connection status

## 🌟 **Major Achievement**

This notebook now demonstrates **fully functional Bing grounding** with real web search capabilities! The agent can:
- ✅ **Search the web in real-time** for current financial market information
- ✅ **Provide up-to-date responses** based on latest market data and news  
- ✅ **Include source citations** from web search results
- ✅ **Maintain financial disclaimers** while leveraging current information

---

*Happy (grounded) agent building!* 🌐🤖